In [1]:
# this notebook will add AFML features
import pandas as pd
import json
from pathlib import Path

def find_project_root(marker=".git"):
    p = Path.cwd()
    while p != p.parent:
        if (p / marker).exists():
            return p
        p = p.parent
    raise FileNotFoundError("Project root not found")

folder_path = find_project_root() / "data" / "mlData"

src_path = folder_path / "raw" / "BTCUSDT-5m-v7-H.jsonl"

# Read JSONL file - keep timestamp as raw number
df = pd.read_json(src_path, lines=True, convert_dates=False)
df.head()

,timestamp,open,high,low,close,vol,atr42,ts_5m,label,train_mask
0,1704037500000,42497.851562,42518.148438,42480.691406,42482.250000,56.512959,NaN,1704037500000,0,0
1,1704037800000,42482.238281,42500.000000,42411.101562,42414.421875,47.354488,NaN,1704037800000,0,0
2,1704038100000,42414.429688,42471.988281,42414.421875,42457.171875,43.228668,NaN,1704038100000,0,0
3,1704038400000,42457.171875,42484.828125,42436.468750,42449.601562,74.418266,NaN,1704038400000,0,0
4,1704038700000,42449.589844,42488.000000,42449.589844,42487.988281,38.818489,NaN,1704038700000,0,0


In [2]:
print(df.columns)

Index(['timestamp', 'open', 'high', 'low', 'close', 'vol', 'atr42', 'ts_5m',
       'label', 'train_mask'],
      dtype='object')


# AMFL features
- FDD windows on close - search for d in

In [3]:
import numpy as np
from statsmodels.tsa.stattools import adfuller
rolling_window = 200

# Fixed-Width Window FFD — causal, no lookahead
def ffd_weights(d, size):
    w = [1.0]
    for k in range(1, size):
        w.append(-w[-1] * (d - k + 1) / k)
    return np.array(w[::-1])  # oldest weight first

def fracdiff_ffd(series, d, window=rolling_window):
    """Returns NaN for first (window-1) rows."""
    w = ffd_weights(d, window)
    out = np.full(len(series), np.nan)
    arr = series.values
    for i in range(window - 1, len(arr)):
        out[i] = np.dot(w, arr[i - window + 1 : i + 1])
    return out

# ADF grid search — find minimum d passing at 95% CI
# let window = 200 (16 hrs)
# Typical result for BTC: minimum d ≈ 0.35/ 100 windows bar
# for d in np.arange(0.3, 0.4, 0.01):
#     fd = fracdiff_ffd(df['close'], d=d, window=rolling_window)
#     valid = fd[~np.isnan(fd)]
#     p = adfuller(valid, autolag='AIC')[1] # strict search
#     print(f"d={d:.3f}  ADF p-value={p:.4f}  {'PASS' if p < 0.05 else 'fail'}")

# ADF search
d=0.300  ADF p-value=0.0900  fail
d=0.310  ADF p-value=0.0806  fail
d=0.320  ADF p-value=0.0716  fail
d=0.330  ADF p-value=0.0629  fail
d=0.340  ADF p-value=0.0546  fail
d=0.350  ADF p-value=0.0469  PASS
d=0.360  ADF p-value=0.0398  PASS
d=0.370  ADF p-value=0.0332  PASS
d=0.380  ADF p-value=0.0274  PASS
d=0.390  ADF p-value=0.0221  PASS
d=0.400  ADF p-value=0.0176  PASS

In [4]:
df['close_fracdiff'] = fracdiff_ffd(df['close'], d=0.35, window=200)
df.head()

,timestamp,open,high,low,close,vol,atr42,ts_5m,label,train_mask,close_fracdiff
0,1704037500000,42497.851562,42518.148438,42480.691406,42482.250000,56.512959,NaN,1704037500000,0,0,NaN
1,1704037800000,42482.238281,42500.000000,42411.101562,42414.421875,47.354488,NaN,1704037800000,0,0,NaN
2,1704038100000,42414.429688,42471.988281,42414.421875,42457.171875,43.228668,NaN,1704038100000,0,0,NaN
3,1704038400000,42457.171875,42484.828125,42436.468750,42449.601562,74.418266,NaN,1704038400000,0,0,NaN
4,1704038700000,42449.589844,42488.000000,42449.589844,42487.988281,38.818489,NaN,1704038700000,0,0,NaN


# Sanity Test
was it really stationary ?

In [5]:
# --- ADF Test ---
series = df['close_fracdiff'].dropna()

adf_stat, p_value, _, _, critical_values, _ = adfuller(series, autolag='AIC')
print("=== ADF Test on close_fracdiff ===")
print(f"  ADF Statistic : {adf_stat:.4f}")
print(f"  p-value       : {p_value:.6f}")
for level, cv in critical_values.items():
    print(f"  Critical Value ({level}): {cv:.4f}")
print()
if p_value < 0.05:
    print("✓ STATIONARY — reject H0 (unit root), p < 0.05")
else:
    print("✗ NOT stationary — fail to reject H0, p >= 0.05")

=== ADF Test on close_fracdiff ===
  ADF Statistic : -2.8867
  p-value       : 0.046920
  Critical Value (1%): -3.4304
  Critical Value (5%): -2.8616
  Critical Value (10%): -2.5668

✓ STATIONARY — reject H0 (unit root), p < 0.05


```
Index(['timestamp', 'open', 'high', 'low', 'close', 'vol', 'atr42', 'ts_5m',
       'label', 'train_mask'],
      dtype='object')

=== ADF Test on close_fracdiff ===
  ADF Statistic : -2.8867
  p-value       : 0.046920
  Critical Value (1%): -3.4304
  Critical Value (5%): -2.8616
  Critical Value (10%): -2.5668

✓ STATIONARY — reject H0 (unit root), p < 0.05
```

In [6]:
# save to JSONL for training
out_path = folder_path / "BTCUSD-5m-v7-augmented.jsonl"

df.to_json(out_path, orient="records", lines=True)
print(f"final shape : {df.shape}")
print(f"Saved {len(df)} rows to {out_path}")


final shape : (228772, 11)
Saved 228772 rows to /Users/aimliu/Library/CloudStorage/OneDrive-Personal/_CODE/Python/py-CandleScience/data/mlData/BTCUSD-5m-v7-augmented.jsonl
